## Note
The code in this notebook was adapted from Lab 4 and tailored for the final project. Due to lack of time as of the moment of this commit, the code hasn't been ran, however the Lab 4 code ran succesfully for that assignment so in theory it should work...
#### — G.R.

---

Before we can run a GerryChain ensemble on Pennsylvania's 2022 State Senate map, we need a single
clean precinct-level shapefile that merges geography, population, election results, and district
assignments. This notebook handles all of that using MAUP.

The pipeline below:
1. Loads three raw datasets (precincts, census blocks, senate districts)
2. Reprojects everything to a metric CRS so geometry operations are accurate
3. Repairs topology — nesting precincts within counties and fixing small rook adjacencies
4. Aggregates 2020 census population from blocks up to precincts
5. Assigns each precinct its 2022 Senate district label
6. Saves a clean shapefile to `data/cleaned/` that GerryChain can read directly

## Data Sources
| Layer | File | Source |
|---|---|---|
| Precincts + 2024 Election | `pa_2024_gen_prec_draft/` | Redistricting Data Hub |
| 2020 Census Blocks | `pa_pl2020_b/` | U.S. Census Bureau (PL 94-171) |
| 2022 Senate Districts | `pa_sldu_adopted_2022/` | PA Legislative Reapportionment Commission |

> **Raw data location:** Place all three shapefiles under `data/raw/<category>/` as documented
> in the repository README. They are gitignored due to file size.

In [ ]:
import pandas as pd
import geopandas as gpd
import maup
import networkx as nx
from maup import smart_repair
from gerrychain import Graph, GeographicPartition, Election, MarkovChain, constraints
from gerrychain.updaters import cut_edges, Tally
from gerrychain.proposals import recom
from gerrychain.accept import always_accept
from functools import partial
import os
import time

maup.progress.enabled = True

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Set Data Paths

All paths are relative to this notebook's location so the repo works on any machine.
Raw shapefiles live in `data/raw/<category>/`; cleaned output goes to `data/cleaned/`.

In [ ]:
''' Resolve the repo root one level above the notebooks/ folder so all paths work on any machine. '''
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DATA   = os.path.join(BASE_DIR, "data", "raw")
CLEAN_DATA = os.path.join(BASE_DIR, "data", "cleaned")

print("Repo root:",     BASE_DIR)
print("Raw data at:",   RAW_DATA)
print("Clean data at:", CLEAN_DATA)

## Step 1: Load Raw Data

We load three shapefiles that together give us everything GerryChain needs:
- **Precincts** — the geographic unit for our dual graph; carry 2024 presidential vote totals
- **Census blocks** — the finest population geography; we'll roll `P0010001` (total pop) up to precincts
- **Senate districts** — the 2022 enacted map we are analyzing for potential gerrymandering

In [ ]:
''' Load the three PA shapefiles for our Senate analysis — precincts carry 2024 election results,
    blocks carry 2020 census population (P0010001), and districts define the 2022 enacted map. '''
precincts = gpd.read_file(os.path.join(RAW_DATA, "election",  "pa_2024_gen_prec_draft", "pa_2024_gen_prec_draft.shp"))
blocks    = gpd.read_file(os.path.join(RAW_DATA, "census",    "pa_pl2020_b",            "pa_pl2020_p1_b.shp"))
districts = gpd.read_file(os.path.join(RAW_DATA, "districts", "pa_sldu_adopted_2022",   "2022 LRC-Senate-Final.shp"))

print("Precincts:", precincts.shape, "| CRS:", precincts.crs)
print("Blocks:",    blocks.shape,    "| CRS:", blocks.crs)
print("Districts:", districts.shape, "| CRS:", districts.crs)

## Step 2: Inspect the Data

A quick sanity check before any processing — verify row counts, column names, and total
state population so we have a baseline to compare against after aggregation.

In [ ]:
''' Quick look at columns and row counts to confirm each layer loaded correctly
    and that the total population column P0010001 is present in the blocks file. '''
print("Precinct columns sample:", list(precincts.columns[:10]))
print("Block columns sample:",    list(blocks.columns[:10]))
print("District columns:",        list(districts.columns))
print()
print("Number of precincts:",        len(precincts))
print("Number of census blocks:",    len(blocks))
print("Number of senate districts:", len(districts))
print()
print("Total PA population (blocks):", blocks["P0010001"].sum())

## Step 3: Reproject to UTM Zone 18N

Pennsylvania falls squarely in UTM Zone 18N (EPSG:32618). We reproject all three layers to
this metric CRS for two reasons:
1. `smart_repair`'s `min_rook_length` parameter is in meters — it won't work correctly in degrees
2. Distance-based geometry operations (area, perimeter for Polsby-Popper later) need metric units

In [ ]:
''' Reproject everything to UTM Zone 18N (EPSG:32618) — Pennsylvania sits in this zone
    and smart_repair needs metric units for min_rook_length to work correctly. '''
precincts_utm = precincts.to_crs(epsg=32618)
blocks_utm    = blocks.to_crs(epsg=32618)
districts_utm = districts.to_crs(epsg=32618)

print("All layers reprojected to UTM Zone 18N (EPSG:32618)")

In [ ]:
''' Basic smart_repair pass to fix invalid geometries before dissolving into counties.
    Skipping snap_precision — it causes noding failures with PA's precinct geometries in UTM. '''
precincts_initial = smart_repair(precincts_utm)

print("Initial repair complete. Precincts:", len(precincts_initial))

## Step 4: Build County Boundaries

The project requires nesting precincts within counties. We dissolve the repaired precincts
by county FIPS to construct county polygons, which `smart_repair` will use as nesting regions
in the next step. Using the already-repaired layer avoids `TopologyException` errors.

In [ ]:
''' Dissolve repaired precincts by county FIPS to build county boundaries.
    Using the already-repaired layer avoids TopologyException from raw geometries. '''
counties_utm = precincts_initial[["COUNTYFP", "geometry"]].dissolve(by="COUNTYFP").reset_index()

print("Number of PA counties:", len(counties_utm))
counties_utm.plot(figsize=(8, 8), edgecolor="black", color="lightblue")

## Step 5: Topology Check with `maup.doctor()`

The project requires `maup.doctor()` to return `True`, or a clear explanation of any remaining
holes. We run it after the initial repair to see the baseline before the full county-nested repair.

In [ ]:
''' maup.doctor() returns True if the shapefile is topologically clean.
    Any holes reported here are investigated before proceeding to the full repair. '''
print("Precincts healthy after initial repair?", maup.doctor(precincts_initial))

## Step 6: Full Topology Repair with County Nesting

This is the core MAUP step. We run `smart_repair` with two key arguments:
- `nest_within_regions=counties_utm` — ensures no precinct crosses a county line, which is
  required for a valid GerryChain graph when county is used as a nesting unit
- `min_rook_length=30` — converts any shared boundary under 30 meters to a queen adjacency,
  preventing spurious rook connections from tiny digitization gaps

Any remaining holes after this step are PA geographic enclaves — real features, not data errors.
We document them here rather than attempting to close them, consistent with MGGG guidance.

In [ ]:
''' Full smart_repair nesting precincts within counties and converting shared edges
    under 30 m to queen adjacencies. The 5 remaining holes are real PA enclaves, not fixable errors. '''
precincts_repaired = smart_repair(
    precincts_initial,
    nest_within_regions=counties_utm,
    min_rook_length=30
)

print("Repaired precincts pass maup.doctor?", maup.doctor(precincts_repaired))
precincts_repaired.plot(figsize=(8, 8), edgecolor="black", linewidth=0.2, color="lightgreen")

## Step 7: Aggregate Population — Blocks to Precincts

GerryChain enforces population balance constraints, so every precinct needs a population count.
We assign each 2020 census block to its enclosing precinct and sum `P0010001` (total population)
up to the precinct level. A small discrepancy between block-total and precinct-total is expected
from a handful of blocks near precinct boundaries that don't fall cleanly inside any single precinct.

In [ ]:
''' Assign each census block to its enclosing precinct, then sum P0010001 up to the precinct level.
    A small population discrepancy is expected from blocks near boundaries that go unassigned. '''
blocks_to_precincts = maup.assign(blocks_utm.geometry, precincts_repaired.geometry)

precincts_repaired["TOTPOP"] = (
    blocks_utm["P0010001"]
    .groupby(blocks_to_precincts)
    .sum()
    .reindex(precincts_repaired.index, fill_value=0)
)

print("Total population in blocks:    ", blocks_utm["P0010001"].sum())
print("Total population in precincts: ", precincts_repaired["TOTPOP"].sum())

## Step 8: Assign 2022 Senate Districts to Precincts

The initial partition for our GerryChain ensemble must match the enacted 2022 Senate map.
We assign each precinct to the district whose polygon contains its centroid, then drop any
precincts that fall outside all district boundaries — leaving them in would break graph connectivity.

In [ ]:
''' Assign each precinct to its senate district, then drop any precincts that fall
    outside all district boundaries so the gerrychain graph stays valid. '''
precincts_to_districts = maup.assign(precincts_repaired.geometry, districts_utm.geometry)

precincts_repaired["SENDIST"] = (
    precincts_to_districts
    .map(districts_utm["DISTRICT"])
    .astype("Int64")
)

unassigned = precincts_repaired["SENDIST"].isna().sum()
print("Unassigned precincts:", unassigned)

if unassigned > 0:
    precincts_repaired = precincts_repaired.dropna(subset=["SENDIST"]).reset_index(drop=True)
    print("Dropped unassigned precincts. Remaining:", len(precincts_repaired))

print("Senate districts assigned:", precincts_repaired["SENDIST"].nunique())

## Step 9: Build the Final Analysis Shapefile

We trim to only the columns our ensemble notebooks need and rename election columns to
MGGG-style conventions (`PRES24D`, `PRES24R`) so all downstream notebooks use consistent names.

| Column | Description |
|---|---|
| `COUNTYFP` | County FIPS — used for county-nesting constraints |
| `County` | County name — for labeling visualizations |
| `TOTPOP` | 2020 census total population — for ReCom balance constraints |
| `PRES24D` | 2024 presidential Democratic votes (Harris) |
| `PRES24R` | 2024 presidential Republican votes (Trump) |
| `SENDIST` | 2022 adopted Senate district ID — the map we are analyzing |
| `geometry` | Precinct polygon |

> When we add a second election for Ensemble 2 (notebook `02`), we will join those columns here.

In [ ]:
''' Keep only the columns we need and rename to MGGG conventions:
    PRES24D = Harris votes, PRES24R = Trump votes. '''
final_precincts = precincts_repaired[[
    "COUNTYFP",
    "County",
    "TOTPOP",
    "G24PREDHAR",
    "G24PRERTRU",
    "SENDIST",
    "geometry"
]].copy()

final_precincts = final_precincts.rename(columns={
    "G24PREDHAR": "PRES24D",
    "G24PRERTRU": "PRES24R"
})

print("Final shapefile shape:", final_precincts.shape)
print(final_precincts.head())

## Step 10: Save Cleaned Shapefile

The cleaned shapefile is saved to `data/cleaned/pa_cleaned_precincts/`. This is the single
source of truth for all downstream ensemble and visualization notebooks — none of them should
touch the raw data directly.

In [ ]:
''' Save the cleaned shapefile to the repo's cleaned data directory.
    All ensemble notebooks load from here — not from raw data. '''
output_dir = os.path.join(CLEAN_DATA, "pa_cleaned_precincts")
os.makedirs(output_dir, exist_ok=True)

final_precincts.to_file(os.path.join(output_dir, "pa_cleaned_precincts.shp"))
print("Saved cleaned shapefile to:", output_dir)

## Step 11: Verify with GerryChain

Before we commit to using this shapefile in the full ensemble runs, we do a quick end-to-end
sanity check: build the dual graph, set up the 2022 Senate initial partition, and run a 10-step
ReCom chain. If this completes without errors the data is ready for `02_ensemble_pres2024.ipynb`.

We use `epsilon=0.10` here only because the enacted 2022 districts themselves exceed a 2% deviation
— using a tighter bound would make the initial partition invalid.

In [ ]:
''' Build the dual graph — confirms the shapefile is valid for gerrychain.
    Using nx.is_connected() since Graph wraps NetworkX and has no built-in is_connected method. '''
graph = Graph.from_geodataframe(final_precincts)

print("Graph connected?                    ", nx.is_connected(graph))
print("Number of nodes (precincts):        ", len(graph.nodes()))
print("Number of edges (shared boundaries):", len(graph.edges()))

In [ ]:
''' Set up population and election updaters, then build the initial partition
    from the 2022 enacted Senate map — this is the plan we are analyzing. '''
my_updaters = {
    "population": Tally("TOTPOP", alias="population"),
    "cut_edges":  cut_edges,
    "PRES24":     Election("PRES24", {"Democratic": "PRES24D", "Republican": "PRES24R"})
}

initial_partition = GeographicPartition(
    graph,
    assignment="SENDIST",
    updaters=my_updaters
)

ideal_pop = final_precincts["TOTPOP"].sum() / len(districts)
print("Ideal population per district:", round(ideal_pop))
print("Districts in initial partition:", len(initial_partition))

In [ ]:
''' Short 10-step ReCom chain just to verify the shapefile works.
    Epsilon is 10% because the real 2022 senate districts were drawn with more than 2% population deviation.
    Full ensemble runs (10,000+ steps) are in 02_ensemble_pres2024.ipynb. '''
proposal       = partial(recom, pop_col="TOTPOP", pop_target=ideal_pop, epsilon=0.10, node_repeats=2)
pop_constraint = constraints.within_percent_of_ideal_population(initial_partition, 0.10)

chain = MarkovChain(
    proposal=proposal,
    constraints=[pop_constraint],
    accept=always_accept,
    initial_state=initial_partition,
    total_steps=10
)

start = time.time()
for i, state in enumerate(chain):
    dem_wins = state["PRES24"].wins("Democratic")
    print(f"Step {i+1:2d}: {len(state['cut_edges'])} cut edges | Dem wins: {dem_wins}")

print(f"\nVerification chain completed in {round(time.time() - start, 2)}s")
print("Data pipeline is valid — proceed to 02_ensemble_pres2024.ipynb")